In [1]:
# ==========================================================
# CORRELATION TABLE: PCs, text features, DSI-SS items, crisis items
# ==========================================================
import os, re, numpy as np, pandas as pd
from scipy.stats import pearsonr

# --- Load raw data to get item-level columns ---
RAW_PATH = os.path.join("..", "data", "raw", "ema_personality_plus_surveys_merged.csv")
FEAT_PATH = os.path.join("..", "data", "processed", "pm_day_features.csv")
OUT_DIR = os.path.join("..", "outputs", "tables")
os.makedirs(OUT_DIR, exist_ok=True)

PID_COL = "expiwell_id_clean"
DT_COL = "ema_dt"

raw = pd.read_csv(RAW_PATH)
if PID_COL not in raw.columns:
    PID_COL = "expiwell_id"
if DT_COL not in raw.columns:
    DT_COL = "start_date"

raw[DT_COL] = pd.to_datetime(raw[DT_COL], errors="coerce")
raw["ema_date"] = raw[DT_COL].dt.date.astype(str)

# Detect DSI-SS items
def find_dsi(df, prefix):
    cols = [c for c in df.columns if c.startswith(prefix)]
    def pick(pats):
        for p in pats:
            hits = [c for c in cols if re.search(p, c, re.IGNORECASE)]
            if hits:
                return sorted(hits, key=len)[0]
        return None
    return {
        "A": pick([r"killing myself"]),
        "B": pick([r"formulated.*plan", r"considered possible ways", r"plans?"]),
        "C": pick([r"control over", r"under my control", r"control"]),
        "D": pick([r"impulses to kill myself", r"impulses"]),
    }

DSI = find_dsi(raw, "dailyPM__")
dsi_items = {k: v for k, v in DSI.items() if v is not None}
print("DSI items found:", list(dsi_items.keys()))

# Detect crisis items (top 5 correlated with total)
PM_CRISIS_TOTAL = "dailyPM__Total Score from 5 Questions"
crisis_item_cols = []
if PM_CRISIS_TOTAL in raw.columns:
    candidates = []
    for c in raw.columns:
        if not c.startswith("dailyPM__") or not pd.api.types.is_numeric_dtype(raw[c]):
            continue
        if c == PM_CRISIS_TOTAL:
            continue
        s = raw[c].dropna()
        if len(s) < 100 or s.nunique() != 5:
            continue
        tmp = raw[[c, PM_CRISIS_TOTAL]].dropna()
        if len(tmp) < 200:
            continue
        corr = tmp.corr().iloc[0, 1]
        candidates.append((c, corr))
    candidates.sort(key=lambda x: -abs(x[1]))
    crisis_item_cols = [c for c, _ in candidates[:5]]
print(f"Crisis items found: {len(crisis_item_cols)}")

# Aggregate items to day level
def first_nonnull(x):
    x = x.dropna()
    return x.iloc[0] if len(x) else np.nan

agg = {}
for label, col in dsi_items.items():
    agg[f"DSI_{label}"] = (col, first_nonnull)
for i, col in enumerate(crisis_item_cols):
    agg[f"SCS_item{i+1}"] = (col, first_nonnull)

raw_day = (
    raw.sort_values([PID_COL, DT_COL])
    .groupby([PID_COL, "ema_date"], as_index=False)
    .agg(**agg)
)
raw_day["ema_date"] = raw_day["ema_date"].astype(str)

# Load features and merge
feat = pd.read_csv(FEAT_PATH)
feat["ema_date"] = feat["ema_date"].astype(str)
merged = feat.merge(raw_day, on=[PID_COL, "ema_date"], how="left")

print(f"Merged: {len(merged)} rows")

# Build correlation matrix
corr_cols = []

# PCs (within-person)
for i in range(1, 6):
    if f"PC{i}_within" in merged.columns:
        corr_cols.append(f"PC{i}_within")

# Text features (within-person)
for f in ["log1p_wc_within", "instability_cosdist_within", "root_ttr_within"]:
    if f in merged.columns:
        corr_cols.append(f)

# DSI items + total
for label in ["A", "B", "C", "D"]:
    col = f"DSI_{label}"
    if col in merged.columns:
        corr_cols.append(col)
if "dsi_PM_total" in merged.columns:
    corr_cols.append("dsi_PM_total")

# Crisis items + total
for i in range(1, 6):
    col = f"SCS_item{i}"
    if col in merged.columns:
        corr_cols.append(col)
if "crisis_PM_from_full" in merged.columns:
    corr_cols.append("crisis_PM_from_full")

print(f"\nCorrelation variables ({len(corr_cols)}):")
for c in corr_cols:
    n_valid = merged[c].notna().sum()
    print(f"  {c}: n={n_valid}")

# Compute correlation matrix
corr_matrix = merged[corr_cols].corr()

# Save
corr_path = os.path.join(OUT_DIR, "correlation_matrix.csv")
corr_matrix.to_csv(corr_path)
print(f"\nSaved: {corr_path}")

# Display
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
print("\n" + corr_matrix.round(3).to_string())

DSI items found: ['A', 'B', 'C', 'D']
Crisis items found: 5
Merged: 2511 rows

Correlation variables (19):
  PC1_within: n=2511
  PC2_within: n=2511
  PC3_within: n=2511
  PC4_within: n=2511
  PC5_within: n=2511
  log1p_wc_within: n=2511
  instability_cosdist_within: n=2511
  root_ttr_within: n=2385
  DSI_A: n=2511
  DSI_B: n=2511
  DSI_C: n=2511
  DSI_D: n=2511
  dsi_PM_total: n=2511
  SCS_item1: n=2510
  SCS_item2: n=2511
  SCS_item3: n=2511
  SCS_item4: n=2511
  SCS_item5: n=2511
  crisis_PM_from_full: n=2511

Saved: ..\outputs\tables\correlation_matrix.csv

                            PC1_within  PC2_within  PC3_within  PC4_within  PC5_within  log1p_wc_within  instability_cosdist_within  root_ttr_within  DSI_A  DSI_B  DSI_C  DSI_D  dsi_PM_total  SCS_item1  SCS_item2  SCS_item3  SCS_item4  SCS_item5  crisis_PM_from_full
PC1_within                       1.000      -0.326      -0.048      -0.035      -0.062           -0.212                       0.491           -0.156  0.181  0.154  0